# KNN Classification## AimImplement K-Nearest Neighbors for classification with configurable k value.

## Problem TypeClassification

## Import Libraries

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

sns.set_style("whitegrid")


## Dataset Input Options

In [ ]:
# Cell A: Load toy dataset
# Keep this as default for quick exam execution.
raw = load_iris()

if hasattr(raw, "frame") and raw.frame is not None:
    df = raw.frame.copy()
    if "target" not in df.columns:
        df["target"] = raw.target
else:
    df = pd.DataFrame(raw.data, columns=raw.feature_names)
    df["target"] = raw.target

TARGET_COLUMN = "target"
DATA_SOURCE = "toy"
print("Using toy dataset. Shape:", df.shape)


In [ ]:
# Cell B: Load local CSV dataset
# Change USE_CSV to True only when you have an exam CSV file.
USE_CSV = False
CSV_PATH = "your_dataset.csv"
TARGET_COLUMN = "target"

if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    DATA_SOURCE = "csv"
    print("Using CSV dataset. Shape:", df.shape)
else:
    print("CSV mode is OFF. Continuing with toy dataset.")


## Dataset Overview / One-cell EDA

In [ ]:
print("Shape:", df.shape)
print("\nHead:")
display(df.head())
print("\nInfo:")
df.info()
print("\nDescribe:")
display(df.describe(include="all"))
print("\nMissing values:")
print(df.isnull().sum())


## Preprocessing (Generic and Reusable)

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

numeric_cols = X.select_dtypes(include=["number"]).columns
categorical_cols = X.select_dtypes(exclude=["number"]).columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

print("Feature shape before preprocessing:", X.shape)


## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


## Apply Algorithm

In [ ]:
k_value = 5
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=k_value)),
    ]
)

model.fit(X_train, y_train)
print(f"KNN (k={k_value}) model trained.")


## Predictions / Results

In [ ]:
y_pred = model.predict(X_test)
display(pd.DataFrame({"Actual": y_test.values[:10], "Predicted": y_pred[:10]}))


## Performance Metrics

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


## Visualization

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## InterpretationModel achieved good classification performance with clear separation in the confusion matrix.

## SuggestionsTune hyperparameters, use cross-validation, and test with larger datasets for stronger generalization.

## ConclusionThis notebook provides a reusable classification workflow for toy and CSV datasets in exam settings.